# Backend-8 (MES 불량 손실시간/재작업시간/Total) 집계 노트북

첨부 사양서(2026-01-30 추가 백엔드-8 설계)에 따라 아래를 계산합니다.

- 이전 공정 MES 불량 손실시간 (`d1_machine_log.mes_fail_wasted_time`)
- MES 불량 재작업시간 (Vision 기준, `e2_vision_ct.vision_op_ct`)
- FCT에서의 MES 불량 재작업시간 (`d1_machine_log.mes_fail2_wasted_time`)
- Total MES 불량 손실시간 = (이전 공정 손실) + (Vision 재작업) + (FCT 재작업)
- 출력 컬럼: `prod_day, shift_type, total_mes_wasted_time, updated_at`

> **중요(사양 반영)**  
> - KST 로컬 기준 주/야간: day=[08:30,20:29:59], night=[20:30,익일08:29:59]  
> - 시간 문자열은 `HH:MI:SS` 로 정규화(소수초는 .5 반올림)  
> - 손실 구간이 shift 경계를 넘으면 초 단위로 분할하여 각 shift에 따로 반영  
> - 다만 MES 불량 **개수(row count)** 는 겹칠 때 더 많은 손실초를 가진 shift 1곳에만 카운트


In [1]:
# (필요 시) 설치
# !pip -q install sqlalchemy psycopg2-binary pandas python-dateutil

In [2]:
from __future__ import annotations

import math
from dataclasses import dataclass
from datetime import datetime, date, time, timedelta
from typing import Dict, Tuple, List, Optional

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import Engine
from zoneinfo import ZoneInfo

KST = ZoneInfo("Asia/Seoul")

DB_CONFIG = {
    "host": "100.105.75.47",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "leejangwoo1!",
}

In [3]:
def make_engine() -> Engine:
    url = (
        f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
        f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}"
    )
    # pool 최소화 + 연결 끊김 대비(pre_ping)
    return create_engine(
        url,
        pool_size=1,
        max_overflow=0,
        pool_pre_ping=True,
        pool_recycle=1800,
    )

def connect_with_retry(sleep_sec: int = 5) -> Engine:
    while True:
        try:
            eng = make_engine()
            # smoke test
            with eng.connect() as conn:
                conn.execute(text("SELECT 1"))
            print("[INFO] DB connected")
            return eng
        except Exception as e:
            print(f"[RETRY] DB connect failed: {e!r}")
            import time as _t
            _t.sleep(sleep_sec)

engine = connect_with_retry()


[INFO] DB connected


In [4]:
# -----------------------------
# 시간 정규화: 'HH:MM:SS.xx' -> 'HH:MM:SS' (소수초 .5 반올림)
# -----------------------------
def _round_half_up(x: float) -> int:
    # .5 -> 올림(half-up)
    return int(math.floor(x + 0.5))

def norm_hms(s: str) -> str:
    """'06:41:23.21' 같은 문자열을 초 단위로 반올림해 '06:41:23'로 변환."""
    s = (s or "").strip()
    if not s:
        return "00:00:00"
    # HH:MM:SS(.fff)
    parts = s.split(":")
    if len(parts) != 3:
        raise ValueError(f"bad time string: {s}")
    hh = int(parts[0])
    mm = int(parts[1])
    ss = float(parts[2])
    sec = _round_half_up(ss)
    # carry
    mm_add, sec = divmod(sec, 60)
    mm += mm_add
    hh_add, mm = divmod(mm, 60)
    hh = (hh + hh_add) % 24
    return f"{hh:02d}:{mm:02d}:{sec:02d}"

def parse_end_day(yyyymmdd: str) -> date:
    return date(int(yyyymmdd[:4]), int(yyyymmdd[4:6]), int(yyyymmdd[6:8]))

def dt_from_endday_time(end_day: str, t_hms: str) -> datetime:
    d = parse_end_day(end_day)
    h, m, s = map(int, t_hms.split(":"))
    return datetime(d.year, d.month, d.day, h, m, s, tzinfo=KST)

# -----------------------------
# Shift window (half-open)
# day:   [D 08:30, D 20:30)
# night: [D 20:30, D+1 08:30)
# -----------------------------
DAY_START = time(8, 30, 0)
DAY_END_EXCL = time(20, 30, 0)   # 20:29:59 포함 -> 20:30 미포함
NIGHT_START = time(20, 30, 0)
NIGHT_END_EXCL = time(8, 30, 0)  # 익일 08:29:59 포함 -> 08:30 미포함

def shift_windows_for_prod_day(prod_day: str) -> Dict[Tuple[str,str], Tuple[datetime, datetime]]:
    d = parse_end_day(prod_day)
    day_s = datetime(d.year, d.month, d.day, 8, 30, 0, tzinfo=KST)
    day_e = datetime(d.year, d.month, d.day, 20, 30, 0, tzinfo=KST)
    night_s = datetime(d.year, d.month, d.day, 20, 30, 0, tzinfo=KST)
    night_e = datetime(d.year, d.month, d.day, 8, 30, 0, tzinfo=KST) + timedelta(days=1)
    return {
        (prod_day, "day"): (day_s, day_e),
        (prod_day, "night"): (night_s, night_e),
    }

def overlap_seconds(a_s: datetime, a_e: datetime, b_s: datetime, b_e: datetime) -> int:
    """half-open [s,e) 구간의 겹치는 초(정수)"""
    s = max(a_s, b_s)
    e = min(a_e, b_e)
    if e <= s:
        return 0
    return int((e - s).total_seconds())

def allocate_interval_to_shifts(start_dt: datetime, end_dt: datetime) -> Dict[Tuple[str,str], int]:
    """임의 구간(half-open [start_dt,end_dt))을 prod_day/shift로 분배"""
    if end_dt <= start_dt:
        return {}
    # 검사 범위에 걸칠 수 있는 prod_day 후보: start_date-1 ~ end_date
    start_date = (start_dt - timedelta(days=1)).date()
    end_date = end_dt.date()
    alloc: Dict[Tuple[str,str], int] = {}
    d = start_date
    while d <= end_date:
        prod_day = f"{d.year:04d}{d.month:02d}{d.day:02d}"
        for key, (ws, we) in shift_windows_for_prod_day(prod_day).items():
            sec = overlap_seconds(start_dt, end_dt, ws, we)
            if sec:
                alloc[key] = alloc.get(key, 0) + sec
        d += timedelta(days=1)
    return alloc

def prod_day_shift_of_ts(ts: datetime) -> Tuple[str, str]:
    """단일 timestamp를 (prod_day, shift_type)로 매핑"""
    t = ts.timetz().replace(tzinfo=None)
    if DAY_START <= t < DAY_END_EXCL:
        d = ts.date()
        return (f"{d.year:04d}{d.month:02d}{d.day:02d}", "day")
    if t >= NIGHT_START:
        d = ts.date()
        return (f"{d.year:04d}{d.month:02d}{d.day:02d}", "night")
    # 00:00~08:29:59 => 전날 night
    d = (ts - timedelta(days=1)).date()
    return (f"{d.year:04d}{d.month:02d}{d.day:02d}", "night")

def format_hms_korean(total_sec: int) -> str:
    total_sec = int(total_sec)
    if total_sec <= 0:
        return "0초"
    h, rem = divmod(total_sec, 3600)
    m, s = divmod(rem, 60)
    parts = []
    if h:
        parts.append(f"{h}시간")
    if m:
        parts.append(f"{m}분")
    if s:
        parts.append(f"{s}초")
    return " ".join(parts)


In [5]:
# -----------------------------
# 1) mes_fail_wasted_time: 이전 공정 불량 손실시간 + MES 불량 개수(행 개수)
# -----------------------------
SQL_MES_FAIL = """
SELECT
  end_day,
  from_time,
  to_time
FROM d1_machine_log.mes_fail_wasted_time
"""

df_mes = pd.read_sql(SQL_MES_FAIL, engine)

# 시간 정규화
df_mes["from_hms"] = df_mes["from_time"].astype(str).map(norm_hms)
df_mes["to_hms"]   = df_mes["to_time"].astype(str).map(norm_hms)

# datetime 변환 (KST)
from_dt = []
to_dt = []
for end_day, f, t in zip(df_mes["end_day"].astype(str), df_mes["from_hms"], df_mes["to_hms"]):
    fdt = dt_from_endday_time(end_day, f)
    tdt = dt_from_endday_time(end_day, t)
    # 자정 넘김 대응: to < from 이면 익일로
    if tdt < fdt:
        tdt = tdt + timedelta(days=1)
    from_dt.append(fdt)
    to_dt.append(tdt)

df_mes["from_dt"] = from_dt
df_mes["to_dt"] = to_dt

# 사양 예시를 만족하기 위해 (from, to] 초를 집계:
#  - 시작 초는 제외, 종료 초는 포함
#  - 이를 half-open [from+1s, to+1s) 로 치환
df_mes["int_s"] = df_mes["from_dt"] + timedelta(seconds=1)
df_mes["int_e"] = df_mes["to_dt"]   + timedelta(seconds=1)

# 각 row를 shift에 분배(손실초)
alloc_rows: List[Dict[Tuple[str,str], int]] = []
for s, e in zip(df_mes["int_s"], df_mes["int_e"]):
    alloc_rows.append(allocate_interval_to_shifts(s, e))

# 손실초 합산
wasted_sec_map: Dict[Tuple[str,str], int] = {}
count_map: Dict[Tuple[str,str], int] = {}
for alloc in alloc_rows:
    # 손실초 분배
    for k, sec in alloc.items():
        wasted_sec_map[k] = wasted_sec_map.get(k, 0) + int(sec)
    # MES 불량 개수는 겹치면 더 큰 sec 쪽 1곳에만
    if not alloc:
        continue
    # 최대 손실초 shift 1곳 선정
    best_k = max(alloc.items(), key=lambda kv: (kv[1], kv[0][1]))[0]  # tie-break: day > night (문자 비교)
    count_map[best_k] = count_map.get(best_k, 0) + 1

df_wasted = (
    pd.DataFrame(
        [
            {"prod_day": k[0], "shift_type": k[1], "mes_wasted_sec": v}
            for k, v in wasted_sec_map.items()
        ]
    )
    .sort_values(["prod_day", "shift_type"])
    .reset_index(drop=True)
)

df_count = (
    pd.DataFrame(
        [
            {"prod_day": k[0], "shift_type": k[1], "mes_fail_count": v}
            for k, v in count_map.items()
        ]
    )
    .sort_values(["prod_day", "shift_type"])
    .reset_index(drop=True)
)

df_prev_process = df_wasted.merge(df_count, on=["prod_day", "shift_type"], how="outer").fillna(0)
df_prev_process["mes_wasted_sec"] = df_prev_process["mes_wasted_sec"].astype(int)
df_prev_process["mes_fail_count"] = df_prev_process["mes_fail_count"].astype(int)

df_prev_process.head()


,prod_day,shift_type,mes_wasted_sec,mes_fail_count
0,20260108,night,6398,103
1,20260109,day,2370,59
2,20260109,night,2240,56
3,20260110,day,749,26
4,20260110,night,3180,53


In [9]:
# -----------------------------
# 2) MES 재작업 시간(Vision): (Vision1_only + Vision2_only) / 4 * MES 불량 개수
#    - month: yyyymm
#    - 입력 prod_day/shift window 기준 '이전달' 사용
# -----------------------------
def prev_month_yyyymm_by_window(prod_day: str, shift_type: str) -> str:
    d = parse_end_day(prod_day)
    if shift_type == "day":
        ws = datetime(d.year, d.month, d.day, 8, 30, 0, tzinfo=KST)
    else:
        ws = datetime(d.year, d.month, d.day, 20, 30, 0, tzinfo=KST)
    first = date(ws.year, ws.month, 1)
    prev_last = first - timedelta(days=1)
    return f"{prev_last.year:04d}{prev_last.month:02d}"

SQL_VISION_PD = """
SELECT month, station, remark, del_out_op_ct_av
FROM e2_vision_ct.vision_op_ct
WHERE month = :month
  AND remark = 'PD'
  AND station IN ('Vision1_only','Vision2_only')
"""

def vision_rework_sec(month: str) -> float:
    """
    (Vision1_only + Vision2_only) / 4
    - 두 행이 모두 있어도 합/4
    - 한 행만 있어도 있는 값만 합산 후 /4 (없는 쪽은 0으로 간주한 효과)
    """
    df = pd.read_sql(text(SQL_VISION_PD), engine, params={"month": month})
    if df.empty:
        return 0.0

    # 숫자 변환(문자/NULL 대비)
    vals = pd.to_numeric(df["del_out_op_ct_av"], errors="coerce").fillna(0.0)

    # 합/4 적용
    return float(vals.sum() / 4.0)

# prod_day/shift별 재작업시간(초)
rows = []
for prod_day, shift_type, mes_fail_count in df_prev_process[["prod_day","shift_type","mes_fail_count"]].itertuples(index=False):
    pm = prev_month_yyyymm_by_window(prod_day, shift_type)
    av = vision_rework_sec(pm)
    rows.append({
        "prod_day": prod_day,
        "shift_type": shift_type,
        "vision_rework_sec": int(round(av * int(mes_fail_count))),
        "vision_prev_month": pm,
        "vision_del_out_op_ct_av_div4": av,  # 계산된 (합/4) 값
    })

df_vision_rework = pd.DataFrame(rows)
df_vision_rework.head()

,prod_day,shift_type,vision_rework_sec,vision_prev_month,vision_del_out_op_ct_av_div4
0,20260108,night,1003,202512,9.7375
1,20260109,day,575,202512,9.7375
2,20260109,night,545,202512,9.7375
3,20260110,day,253,202512,9.7375
4,20260110,night,516,202512,9.7375


In [10]:
# -----------------------------
# 3) FCT 재작업 시간: mes_fail2_wasted_time의 final_ct 합
#    - end_day/end_time 기준 shift 매핑 후 합산
# -----------------------------
SQL_MES_FAIL2 = """
SELECT
  barcode_information,
  end_day,
  end_time,
  final_ct
FROM d1_machine_log.mes_fail2_wasted_time
"""

df_fct = pd.read_sql(SQL_MES_FAIL2, engine)

# end_time 정규화(문자/시간 타입 모두 대응)
def norm_end_time(x) -> str:
    s = str(x).strip()
    # 이미 'HH:MM:SS'면 그대로
    if len(s) >= 8 and s[2] == ":" and s[5] == ":":
        # 소수초가 붙어도 norm_hms가 처리
        return norm_hms(s)
    return norm_hms(s)

df_fct["end_hms"] = df_fct["end_time"].map(norm_end_time)
df_fct["end_dt"] = [
    dt_from_endday_time(str(d), hms)
    for d, hms in zip(df_fct["end_day"].astype(str), df_fct["end_hms"])
]

df_fct["prod_day"] = [prod_day_shift_of_ts(ts)[0] for ts in df_fct["end_dt"]]
df_fct["shift_type"] = [prod_day_shift_of_ts(ts)[1] for ts in df_fct["end_dt"]]

df_fct_rework = (
    df_fct.groupby(["prod_day","shift_type"], as_index=False)["final_ct"]
    .sum()
    .rename(columns={"final_ct":"fct_rework_sec"})
)

# final_ct가 ms일 수도 있으니, 단위가 초가 맞는지 확인 필요
df_fct_rework["fct_rework_sec"] = df_fct_rework["fct_rework_sec"].astype(float)

df_fct_rework.head()


,prod_day,shift_type,fct_rework_sec
0,20230901,day,0.0
1,20230905,day,0.0
2,20230906,day,0.0
3,20240206,day,0.0
4,20240207,day,0.0


In [11]:
# -----------------------------
# 4) Total MES 불량 손실 시간 = 이전 공정 손실 + Vision 재작업 + FCT 재작업
# -----------------------------
df = (
    df_prev_process
    .merge(df_vision_rework[["prod_day","shift_type","vision_rework_sec"]], on=["prod_day","shift_type"], how="left")
    .merge(df_fct_rework, on=["prod_day","shift_type"], how="left")
)

df["vision_rework_sec"] = df["vision_rework_sec"].fillna(0).astype(int)
df["fct_rework_sec"] = df["fct_rework_sec"].fillna(0).astype(float)

df["total_mes_wasted_sec"] = df["mes_wasted_sec"] + df["vision_rework_sec"] + df["fct_rework_sec"]
df["total_mes_wasted_sec"] = df["total_mes_wasted_sec"].astype(int)

df["total_mes_wasted_time"] = df["total_mes_wasted_sec"].map(format_hms_korean)

df["updated_at"] = datetime.now(tz=KST)

out = df[["prod_day","shift_type","total_mes_wasted_time","updated_at"]].sort_values(["prod_day","shift_type"]).reset_index(drop=True)
out.head(50)


,prod_day,shift_type,total_mes_wasted_time,updated_at
0,20260108,night,2시간 11분 21초,2026-01-30 11:57:20.049564+09:00
1,20260109,day,54분 2초,2026-01-30 11:57:20.049564+09:00
2,20260109,night,51분 53초,2026-01-30 11:57:20.049564+09:00
3,20260110,day,18분 34초,2026-01-30 11:57:20.049564+09:00
4,20260110,night,1시간 11분 11초,2026-01-30 11:57:20.049564+09:00
5,20260111,day,6분 57초,2026-01-30 11:57:20.049564+09:00
6,20260112,day,3시간 1분 34초,2026-01-30 11:57:20.049564+09:00
7,20260112,night,2시간 53분 45초,2026-01-30 11:57:20.049564+09:00
8,20260113,day,2시간 25분 47초,2026-01-30 11:57:20.049564+09:00
9,20260113,night,1시간 19분 10초,2026-01-30 11:57:20.049564+09:00


## 메모 / 점검 포인트

1. `mes_fail_wasted_time`의 `from_time/to_time`이 **동일 end_day 기준**으로 기록되며 자정 넘어가는 케이스가 있으면 `to < from`일 때 익일로 보정합니다.  
2. 사양 예시(`08:28:59~08:30:00 => night 60초 / day 1초`)를 맞추기 위해 손실초는 **(from, to]** 기준(시작초 제외/종료초 포함)으로 처리했습니다.  
3. `mes_fail2_wasted_time.final_ct`의 단위가 초인지(ms인지) DB 정의를 확인하세요(현재는 초로 가정).  
